#### Install Homebrew (for Linux / MacOs unix)

In [ ]:
# Run this cell
!/bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

# Note: You can also run in the terminal but without the preceeding !
# /bin/bash -c "$(curl -fsSL https://raw.githubusercontent.com/Homebrew/install/HEAD/install.sh)"

In [2]:
# add homebrew to path and apply it to the current session
# For Linux
!echo 'eval "$(/home/linuxbrew/.linuxbrew/bin/brew shellenv)"'  >> ~/.bashrc
!eval "$(/home/linuxbrew/.linuxbrew/bin/brew shellenv)"

# For macOS M1/M2/M3 
# echo 'eval "$(/opt/homebrew/bin/brew shellenv)"' >> ~/.zprofile 
# eval "$(/opt/homebrew/bin/brew shellenv)"

# For masOS (Intel)
# echo 'eval "$(/usr/local/bin/brew shellenv)"'  >> ~/.zprofile
# eval "$(/usr/local/bin/brew shellenv)"

In [ ]:
# install homebrew dependencies
# For Debian or Ubuntu
!sudo apt-get install build-essential procps curl file git

# For Fedora
# sudo dnf group install development-tools
# sudo dnf install procps-ng curl file

# For CentOS Stream or RHEL
# sudo dnf group install 'Development Tools'
# sudo dnf install procps-ng curl file

# For Arch Linux
# sudo pacman -S base-devel procps-ng curl file git

### Install KitOps CLI (For Linux/macOS)

In [ ]:
!brew tap kitops-ml/kitops
!brew install kitops

In [ ]:
# check installation
!kit version

### Install the required Python library

In [ ]:
# we assume you have Python 3.11 or later already installed
!pip install kitops huggingface_hub

### Unsloth Studio installation

In [ ]:
# curl -fsSL https://unsloth.ai/install.sh | sh


# Then start the studio with the command below and open http://127.0.0.1:8888 in your browser.
# unsloth studio -H 127.0.0.1 -p 8888

### Authenticate KitOps CLI with Jozu Hub

In [ ]:
# Run the below command in the terminal, you will be prompted to insert the username (email) and password, 
# you used to create the jozu hub account. 
# Note: As you are typing the password it won't show on most terminals so don't be surprised 
# just hit enter when you are done.


# kit login jozu.ml

### Pulling a model from HuggingFace


In [ ]:
from huggingface_hub import hf_hub_download

# Will download the quantized GGUF to the given local directory
hf_hub_download(
    repo_id="unsloth/Qwen3.5-9B-GGUF", 
    filename="Qwen3.5-9B-Q4_0.gguf",
    local_dir="./Qwen3.5-9B-Q4"
)

### Writing the model's Kitfile

In [ ]:
from kitops.modelkit.kitfile import Kitfile

# Create new Kitfile
kitfile = Kitfile()

# Set basic metadata
kitfile.manifestVersion = "1.0"
kitfile.package = {
    "name": "Qwen3.5-9B-4Bit-GGUF",
    "version": "1.0",
    "description": "Kitfile for Qwen3.5-9B-Q4_0-GGUF"
}

# Configure model information
kitfile.model = {
    "name": "Qwen3.5-9B-4Bit-GGUF",
    "path": "Qwen3.5-9B-Q4/Qwen3.5-9B-Q4_0.gguf", # path to the GGUF file.
    "version": "1.0",
    "license": "Apache 2.0",
    "description": "4Bit GGUF file of Qwen 3.5 9B"
}


# You can also add other information like below:

# kitfile.code = [
#     {
#         "path": "...",
#         "description": "..."
#     }
# ]

# kitfile.datasets = [
#     {
#         "name": "dataset",
#         "path": "data/sample.csv",
#         "description": "full dataset",
#         "license": "Apache 2.0"
#     }
# ]

# kitfile.docs = [
#     {"path": "docs/README.md"},
#     {"path": "docs/LICENSE"}
# ]

# For more information on what you can add, see https://kitops.org/docs/pykitops/how-to-guides/


# You Can Save the Kitfile locally (Note: It is the Kitfile that specifies how the Modelkit will be bundled)
kitfile.save("Kitfile")


### Pushing the ModelKit to Jozu Hub and signing it

In [ ]:
from kitops.modelkit.manager import ModelKitManager, UserCredentials

# Configure the ModelKit manager
# Note: The email prefix of jack123@gmail.com is just jack123

modelkit_tag = "jozu.ml/email-prefix-here/qwen3.5_9b_4bit_gguf:latest" # all in lowercase
manager = ModelKitManager(
    working_directory=".",
    modelkit_tag=modelkit_tag,
    user_credentials=UserCredentials('full-email-here', 'password-here', namespace='qwen3.5_9b_4bit_gguf')
)

# Assign your Kitfile
manager.kitfile = kitfile

# Pack and push to Jozu Hub
manager.pack_and_push_modelkit(save_kitfile=True)

In [ ]:
# To sign the ModelKit, first install Cosign (an open-source CLI tool from Sigstore used 
# to digitally sign and verify container images)

# brew install cosign

# Then check that the installation was successful
# cosign version

In [ ]:
# Now authenticate Cosign to Jozu Hub so it can push the signature to the registry.

# cosign login jozu.ml -u your-full-email-here -p your-jozu-password-here

In [ ]:
# Now we need to generate the key pair we will use for signing the ModelKit:

# cosign generate-key-pair

In [ ]:
# Finally, we sign the ModelKit with the key:

# cosign sign --key cosign.key jozu.ml/email-prefix-here/qwen3.5_9b_4bit_gguf:latest

### Unpacking the ModelKit from Jozu Hub to a local directory.

In [ ]:
# Since our ModelKit is hosted on Jozu Hub, we can pull it from any machine or environment. 
# To demonstrate this, lets unpack the ModelKit to a models folder in our home directory:

# mkdir ~/models
# cd ~/models

In [ ]:
# Next, we’ll pull and unpack the agent skills to the models directory

# kit pull jozu.ml/email-prefix-here/qwen3.5_9b_4bit_gguf:latest
# kit unpack jozu.ml/email-prefix-here/qwen3.5_9b_4bit_gguf:latest -d ./